# Bridge — Bass `bass_model.onnx` (Magenta distillation)

**Teacher:** MusicVAE `cat-mel_2bar_big` — decode → shift to bass register → 64-float Bridge target.

**Flow:** seed MIDI / random kick+context → `distill_batch_bass` → **BassStudentMLP** → ONNX.

In [ ]:
!pip -q install "tensorflow>=2.14,<2.17" magenta note-seq pretty_midi music21 torch onnx onnxruntime numpy scipy matplotlib

import os, sys
REPO = "/content/Bridge"
sys.path.insert(0, os.path.join(REPO, "ml", "training") if os.path.isdir(REPO) else ".")
import numpy as np, torch
print("Torch", torch.__version__)

In [ ]:
n_samples = 500
TRAIN_STEPS = 1200
BATCH = 64
LR = 1e-3
CACHE_DIR = "/content/magenta_ckpt"
USER_MIDI = None

In [ ]:
from magenta_data_utils import ensure_simple_groove_fixture
midi_path = USER_MIDI if USER_MIDI else ensure_simple_groove_fixture()
print("Seed:", midi_path)

In [ ]:
from magenta_pipeline import MagentaBassTeacher, distill_batch_bass
teacher = MagentaBassTeacher(cache_dir=CACHE_DIR)
X, Y = distill_batch_bass(teacher, n_samples=n_samples, seed=42, midi_path=midi_path)
print("X", X.shape, "Y", Y.shape)

In [ ]:
from bass_onnx_student import train_student_from_xy, export_bass_onnx, BASS_ONNX_IN, BASS_ONNX_OUT
model = train_student_from_xy(X, Y, steps=TRAIN_STEPS, batch=BATCH, lr=LR)
out_dir = os.path.join(REPO, "ml", "models") if os.path.isdir(REPO) else "ml/models"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "bass_model.onnx")
export_bass_onnx(model, out_path)
print("Saved", out_path, BASS_ONNX_IN, "->", BASS_ONNX_OUT)

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession(out_path, providers=["CPUExecutionProvider"])
x = np.random.rand(1, BASS_ONNX_IN).astype(np.float32)
y = sess.run(None, {"input": x})[0]
assert y.shape == (1, BASS_ONNX_OUT)

In [ ]:
# === 7) Perceptual quality (quick MIDI metrics) ===
import os, tempfile
import numpy as np
import onnxruntime as ort
from onnx_runtime_inputs import build_fixed_input
from magenta_data_utils import bridge_student_output_to_pretty_midi
from quality_metrics import perceptual_verdict, score_midi_file

INSTRUMENT = "bass"
sess = ort.InferenceSession(out_path, providers=["CPUExecutionProvider"])
_in = sess.get_inputs()[0]
knobs = np.array([0.5] * 10, dtype=np.float32)
x = build_fixed_input(INSTRUMENT, knobs).reshape(1, -1).astype(np.float32)
y = sess.run(None, {_in.name: x})[0].reshape(-1)
pm = bridge_student_output_to_pretty_midi(INSTRUMENT, y, root_note=0, octave=3, bpm=120.0)
fd, tmp = tempfile.mkstemp(suffix=".mid"); os.close(fd)
pm.write(tmp)
metrics = score_midi_file(tmp, INSTRUMENT, bpm=120.0)
os.unlink(tmp)
print("Perceptual metrics:")
for k in ("rhythmic_consistency", "pitch_entropy", "velocity_range", "density", "groove_score", "melodic_contour_smoothness"):
    print(f"  {k}: {metrics.get(k)}")
print(perceptual_verdict(metrics, INSTRUMENT))

In [ ]:
try:
    from google.colab import files
    files.download(out_path)
except ImportError:
    print(out_path)